# Tutorial 22 -- Parameter Sweeps and Batch Simulation

Use `prepare_simulation(...)` and `simulate_batch(...)` to reuse one compiled schedule across multiple initial states and compare the resulting observables.

**Prerequisites.** Tutorials 07 and 21 are recommended first.


## 1. Goal

We will prepare one spectroscopy-style session and run it across several initial cavity Fock states without rebuilding the Hamiltonian or timeline each time.


## 2. Physical Background

The prepared-session path is useful when the model, drive mapping, and compiled schedule stay fixed while the initial state changes. This is common in scan-style workloads and small inner loops.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    cross_kerr_conditional_phase,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    gaussian_quasistatic_echo_excited_population,
    gaussian_quasistatic_ramsey_excited_population,
    ns,
    ramsey_population,
    resonant_drive_excited_population,
    t1_relaxation_population,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
initial_levels = [0, 1, 2, 3]
dt = 4.0 * ns
probe_duration = 1.0 * us


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=0.0,
    chi=MHz(-2.2),
    kerr=0.0,
    n_cav=8,
    n_tr=2,
)
frame = FrameSpec(omega_c_frame=model.omega_c, omega_q_frame=model.omega_q)
target_detuning = manifold_transition_frequency(model, 0, frame=frame)


## 6. Pulse / Sequence Construction


In [ ]:
probe = Pulse(
    "q",
    0.0,
    probe_duration,
    square_envelope,
    amp=MHz(0.08),
    carrier=carrier_for_transition_frequency(target_detuning),
    label="batch_probe",
)
compiled = SequenceCompiler(dt=dt).compile([probe], t_end=probe_duration + dt)
session = prepare_simulation(
    model,
    compiled,
    {"q": "qubit"},
    config=SimulationConfig(frame=frame, max_step=dt),
)
initial_states = [model.basis_state(0, n) for n in initial_levels]


## 7. Running the Simulation


In [ ]:
batch_results = simulate_batch(session, initial_states, max_workers=1)
final_populations = [final_expectation(result, "P_e") for result in batch_results]
for n, p_e in zip(initial_levels, final_populations, strict=True):
    print(f"initial cavity level n = {n}: final P_e = {p_e:.4f}")


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.bar([str(n) for n in initial_levels], final_populations, color="tab:blue")
ax.set_xlabel("Initial cavity Fock level")
ax.set_ylabel(r"Final $P_e$")
ax.set_title("Batch execution with one prepared spectroscopy session")
plt.show()


## 9. Physical Interpretation

Because the probe is resonant with the `n = 0` qubit line, the higher-Fock initial states respond less strongly. The key workflow point is that the simulation session is reused while only the initial state changes.


## 10. Exercises / Next Steps

- Wrap the prepared-session pattern in a Python loop over several detunings to build a larger sweep efficiently.
- Re-run the batch in a three-mode model and compare storage versus readout conditioning.
- Continue to Tutorial 23 for fitting and convenience calibration-target helpers.
